# 作业 3.1：从标准源能谱提取 HPGe 探测器性能


## 标准源与测量能谱

Eurica gamma 探测阵列由 12 个 Euroball Cluster 组成，每个 Cluster 含 7 个 HPGe 晶体；标定源距探测器约 22 cm。数据已对每个 Cluster 的 7 个晶体作 add-back，再合并 12 个 Cluster，用于考察阵列的整体性能。装置详情见 [Installation and commissioning of EURICA – Euroball-RIKEN Cluster Array](https://www.sciencedirect.com/science/article/pii/S0168583X13003182)。

<img src="eurica.png" alt="Eurica detector array" style="max-width:36%;" />

能谱由 $^{152}$Eu 与 $^{133}$Ba 标准源测得，测量开始于 2013 年 2 月 13 日，记录时长为 7442 s。两个源的参考日期均为 1998 年 1 月 1 日；参考活度分别为 $^{152}$Eu 40.9 kBq（5%）和 $^{133}$Ba 42.2 kBq（3%）。活度衰变修正采用 $T_{1/2}(^{152}\mathrm{Eu})=13.517$ y、$T_{1/2}(^{133}\mathrm{Ba})=3849.3$ d。

<div class="source-line-grid">
<table>
<thead><tr><th>Nuclide</th><th><i>E</i><sub>γ</sub> (keV)</th><th><i>P</i><sub>γ</sub> (%)</th></tr></thead>
<tbody>
<tr><td><sup>133</sup>Ba</td><td>80.9979</td><td>34.06</td></tr>
<tr><td><sup>152</sup>Eu</td><td>121.7817</td><td>28.41</td></tr>
<tr><td><sup>152</sup>Eu</td><td>244.6974</td><td>7.55</td></tr>
<tr><td><sup>133</sup>Ba</td><td>276.3989</td><td>7.164</td></tr>
<tr><td><sup>133</sup>Ba</td><td>302.8508</td><td>18.33</td></tr>
<tr><td><sup>152</sup>Eu</td><td>344.2785</td><td>26.59</td></tr>
</tbody>
</table>
<table>
<thead><tr><th>Nuclide</th><th><i>E</i><sub>γ</sub> (keV)</th><th><i>P</i><sub>γ</sub> (%)</th></tr></thead>
<tbody>
<tr><td><sup>133</sup>Ba</td><td>356.0129</td><td>62.05</td></tr>
<tr><td><sup>152</sup>Eu</td><td>778.9045</td><td>12.93</td></tr>
<tr><td><sup>152</sup>Eu</td><td>867.378</td><td>4.23</td></tr>
<tr><td><sup>152</sup>Eu</td><td>964.079</td><td>14.51</td></tr>
<tr><td><sup>152</sup>Eu</td><td>1112.076</td><td>13.67</td></tr>
<tr><td><sup>152</sup>Eu</td><td>1408.013</td><td>20.87</td></tr>
</tbody>
</table>
</div>

<style>
.source-line-grid { display:grid; grid-template-columns:repeat(2,minmax(0,1fr)); gap:1rem; align-items:start; }
.source-line-grid table { width:100%; margin:0; }
.source-line-grid th:nth-child(n+2), .source-line-grid td:nth-child(n+2) { text-align:right; }
@media (max-width:720px) { .source-line-grid { grid-template-columns:1fr; } }
</style>

能量、$P_\gamma$ 和半衰期以推荐核数据为准，可查阅 [DDEP/LNHB recommended decay data](https://www.lnhb.fr/home/nuclear-data/)。$P_\gamma$ 是每次母核衰变发射该 gamma ray 的概率，不是源活度；它可辅助指认谱线，但不能直接当作实验 peak area 之比。

本作业使用 [gamma.root](gamma.root) 中的 `TH1F h0`。横轴尚未刻度，单位为 channel；每个 bin 宽 0.2 channel。下面是本次测量的合并能谱。纵轴采用 log scale，使强峰和弱峰能够同时显示。

<img src="standard_source_spectrum.png" alt="measured Eu-152 and Ba-133 spectrum" style="max-width:62%;" />

对照标准源的参考能谱与上表，可以先辨认较强的 gamma line，再利用初步线性刻度寻找其余 peak。

<div style="display:flex; flex-wrap:wrap; gap:1rem; align-items:center;">
  <img src="../calibration_method/152Eu.png" alt="Eu-152 reference spectrum" style="max-width:34%; height:auto;" />
  <img src="../calibration_method/133Ba.png" alt="Ba-133 reference spectrum" style="max-width:37%; height:auto;" />
</div>


本作业从标准源测得的 gamma 能谱中提取 HPGe 探测器的三个性能参数：能量刻度、能量分辨率和 full-energy peak efficiency。三者都使用 gamma peak，但所需的观测量和提取方法不同。

| 观测量 | 本作业采用的方法 | 用途 |
| --- | --- | --- |
| peak centroid $\mu$ | local peak fit | energy calibration |
| $\sigma$、FWHM | local peak fit | resolution calibration |
| 孤立峰的 $N_{\rm net}$ | ROI summation $-$ local continuum | efficiency calibration |
| 重叠峰的各自面积 | fitting / deconvolution | 进阶分析 |

本作业的主线是

$$
\boxed{
\begin{aligned}
\text{peak fit}&\rightarrow \mu,\ \mathrm{FWHM},\\
\text{ROI summation}&\rightarrow N_{\rm net},\\
(\mu,E_\gamma)&\rightarrow \text{energy calibration},\\
\frac{N_{\rm net}}{A(t)P_\gamma t_{\rm live}}
&\rightarrow \text{efficiency calibration}.
\end{aligned}}
$$


## Gamma peak 的位置与宽度：local fitting

孤立、近似对称的 gamma peak 可先用 Gaussian signal 描述：

$$
s(x)=H\exp\!\left[-\frac{(x-\mu)^2}{2\sigma^2}\right].
$$

在足够窄的拟合区间内，连续本底可先作 linear approximation：

$$
b(x)=b_0+b_1(x-x_0),\qquad f(x)=s(x)+b(x).
$$

先在 visible peak 之外选择左右 sideband，拟合直线并获得 $b_0$、$b_1$ 的初值；再在完整局部区间内同时拟合 Gaussian 与 linear background。第二步不固定本底参数，因此是 signal 与 background 的 joint fit。本作业的数据是 histogram 计数，实例使用 binned Poisson likelihood。

fit 的主要输出是

$$
\mu,\qquad \sigma,\qquad
\mathrm{FWHM}=2\sqrt{2\ln2}\,\sigma\simeq2.355\sigma.
$$

除 fit status 外，还要检查 Pearson residual

$$
r_i=\frac{n_i-\nu_i}{\sqrt{\nu_i}},
$$

其中 $n_i$ 是观测计数，$\nu_i$ 是模型给出的期望计数。连续的 residual 结构表示当前 peak 或 background model 尚未描述完数据。

867.378 keV peak 用于演示这一过程：左图仅用 sidebands 确定本底初值；右图是在完整局部区间内的 joint fit。这里采用 linear y，以直接观察 peak 两侧的本底。

<div class="peak-example-grid">
  <figure>
    <img src="fit_867_background.png" alt="linear fit to sidebands near the 867.378 keV peak" />
    <figcaption>左右 sideband 的 linear fit。</figcaption>
  </figure>
  <figure>
    <img src="fit_867_linear.png" alt="867.378 keV gamma peak with a linear background" />
    <figcaption>Gaussian + linear background 的 joint fit 与 residual。</figcaption>
  </figure>
</div>

<style>
.peak-example-grid { display:grid; grid-template-columns:repeat(2,minmax(0,1fr)); gap:1rem; max-width:78%; margin:1rem 0; align-items:start; }
.peak-example-grid figure { margin:0; }
.peak-example-grid img { width:100%; height:auto; }
.peak-example-grid figcaption { margin-top:.4rem; font-size:.92rem; line-height:1.35; }
@media (max-width:720px) { .peak-example-grid { grid-template-columns:1fr; max-width:100%; } }
</style>


## 孤立峰的面积：ROI summation 与局域本底

fit 给出的 centroid 和 FWHM 为选取 peak ROI 与 sidebands 提供位置和尺度。令

$$W=\mathrm{FWHM}(\mu).$$

本页的 867.378 keV 示例采用

$$
\text{peak ROI}:\quad \mu-1.5W<x<\mu+1.5W,
$$

$$
\text{left sideband}:\quad \mu-3W<x<\mu-2W,
\qquad
\text{right sideband}:\quad \mu+2W<x<\mu+3W.
$$

这些倍数是教学示例，不是唯一标准。窗口应以局部 FWHM 为尺度，并避开邻近 peak 或明显的 continuum 结构。

设 peak ROI 的 gross counts 为 $G$，左右 sideband 的总计数分别为 $L$、$R$，包含的 bin 数分别为 $m_L$、$m_R$，peak ROI 含 $n$ 个 bins。对关于 peak 对称的窗口，linear continuum 在 peak ROI 下的计数估计为

$$
B=\frac{n}{2}\left(\frac{L}{m_L}+\frac{R}{m_R}\right),
$$

因此

$$
\boxed{N_{\rm net}=G-B.}
$$

若三个区域互不重叠，且 bin counts 服从 Poisson statistics，则

$$
\boxed{
u^2(N_{\rm net})=
G+\left(\frac{n}{2m_L}\right)^2L
+\left(\frac{n}{2m_R}\right)^2R.
}
$$

第一项来自 peak ROI 的计数涨落，后两项来自用 sidebands 估计本底的计数涨落。

<img src="fit_867_integration.png" alt="FWHM-scaled peak ROI and sidebands for the 867.378 keV peak" style="max-width:62%;" />

对本作业中的孤立刻度峰，使用这种 summation 方法求面积，峰形依赖较弱；fit 则用于确定 peak 位置和宽度。若多个 peak 重叠，单个 ROI 无法分开各峰的贡献，才需要 fitting / deconvolution 求各自面积。


## 从 peak 观测量得到探测器性能

### 能量刻度

用各参考线的 centroid $ch_i$ 与已知能量 $E_i$ 拟合

$$E(ch)=a_0+a_1ch.$$

先由两个相隔较远、指认可靠的 peak 建立粗略线性关系，再据此定位其余参考线。最终用 calibration residual

$$\Delta E_i=E_{i,\mathrm{ref}}-E_{\mathrm{cal}}(ch_i)$$

检查刻度；只有 residual 呈现系统曲率时才考虑加入 $a_2ch^2$。把刻度应用到完整 histogram 后，变换前后的总计数应保持一致。

### 能量分辨率

由能量刻度的局部斜率把 channel 上的 width 换算为

$$
FWHM(E)=2\sqrt{2\ln2}\left|\frac{dE}{dch}\right|\sigma_{ch}
\approx2.355\left|\frac{dE}{dch}\right|\sigma_{ch}.
$$

HPGe 的 FWHM 随能量变化常用

$$FWHM(E)=\sqrt{A+BE+CE^2}$$

作经验描述；常数项、$E$ 项和 $E^2$ 项分别概括电子学噪声、载流子统计和随能量增长的电荷收集等贡献。得到的 $FWHM(E)$ 既描述探测器分辨率，也可作为全谱选取 peak ROI 与 sidebands 的自然尺度。

<img src="../calibration_method/width.png" alt="a typical HPGe resolution curve" style="max-width:38%;" />


### Full-energy peak efficiency

把参考活度 $A_0$ 从日期 $t_0$ 修正到测量日期 $t$：

$$
A(t)=A_0\,2^{-(t-t_0)/T_{1/2}}.
$$

由局域本底扣除后的 net peak area 计算

$$
\boxed{
\varepsilon(E_\gamma)=
\frac{N_{\rm net}}{A(t)P_\gamma t_{\rm live}}.
}
$$

若输入量相互独立，且 live-time uncertainty 可忽略，则单个 efficiency point 的相对误差可写为

$$
\left(\frac{u_\varepsilon}{\varepsilon}\right)^2=
\left(\frac{u_N}{N_{\rm net}}\right)^2+
\left(\frac{u_A}{A(t)}\right)^2+
\left(\frac{u_P}{P_\gamma}\right)^2.
$$

同一标准源的活度误差会同时移动该源的全部 efficiency 点，是相关的 normalization uncertainty，不是彼此独立的 point-to-point fluctuation。

若尚未修正 dead time、true-coincidence summing、源几何、衰减与自吸收，这里得到的是该测量设置下的 apparent full-energy peak efficiency。实例暂把题目给出的 7442 s 作为 $t_{\rm live}$；若采集记录区分 real time 与 live time，应使用 live time。

HPGe 的 efficiency 在低能端会因端帽、死层和源封装材料的吸收而下降，在中间能区达到最大值后再随能量升高而下降。令

$$u=\ln\!\left(\frac{E}{100\ \mathrm{keV}}\right),$$

本作业采用经验函数

$$
\varepsilon(E)=\exp\!\left[
p_0+p_1u+p_2u^2-p_3\left(\frac{100\ \mathrm{keV}}{E}\right)^3
\right],\qquad p_3\ge0.
$$

曲线在 log–log 坐标上显示并检查 residual。本数据最低参考能量为 81 keV，更低能区没有刻度点约束。

<img src="../calibration_method/eff.png" alt="a typical HPGe full-energy peak efficiency curve" style="max-width:38%;" />


## 进阶：简单 singlet 方法失效时

linear background 只是在窄区间内对 smooth continuum 的一阶近似。实际 gamma spectrum 还可能包含 Compton continuum 或 edge、邻近 peak、低能 tail 和电子学响应。

- 若 peak 前后的 background level 存在明显 step，可尝试在 linear term 上加入 $\frac{S}{2}\operatorname{erfc}[(x-\mu)/(\sqrt2\sigma)]$；但只有 residual 明显改善时才需要增加参数。本数据的 1408 keV peak 试算没有得到实质改善，因此主分析仍采用 linear background。
- 非 Gaussian peak 可在响应函数中加入 low-energy tail。
- 对 multiplet，模型写成 $f(x)=\sum_k s_k(x)+b(x)$，各 peak area 需要由 fitting / deconvolution 分离。

[ORTEC GammaVision 用户手册](https://www.ortec-online.com/-/media/ametekortec/manuals/a/a66-mnl.pdf?la=en)以 straight-line background 作为基本 ROI/singlet 处理，并在谱形需要时提供 stepped 或 parabolic background。[肖石良等（2024）](https://wulixb.iphy.ac.cn/pdf-content/10.7498/aps.73.20231980.pdf)对更复杂的在线 gamma spectrum 分别加入低能 tail、`erfc` step、polynomial background 和 Compton-edge response；单个 `erfc` 项并不能解释所有本底结构。

### Alternative: fitted response function 的面积

若 signal model 为

$$s(x)=H\exp[-(x-\mu)^2/(2\sigma^2)],$$

且 histogram bin width 为 $w$，则 fitted Gaussian 的总面积为

$$
N_{\rm fit}=\frac{H\sigma\sqrt{2\pi}}{w}.
$$

因为 $H$ 与 $\sigma$ 来自同一次 fit，二者通常相关：

$$
u_N^2=\left(\frac{\sqrt{2\pi}}{w}\right)^2
\left[\sigma^2u_H^2+H^2u_\sigma^2+
2H\sigma\operatorname{Cov}(H,\sigma)\right].
$$

这适用于 multiplet 或更完整的 response-function analysis。对本作业中的孤立峰，ROI summation 是更直接的主方法；$N_{\rm fit}$ 可作为进阶比较。

### Fitted efficiency curve 在给定能量处的误差

若要给出固定能量 $E_0$ 处 fitted efficiency 的误差，令 $q=(100\ \mathrm{keV}/E_0)^3$，则

$$
\mathbf J(E_0)=\varepsilon(E_0)(1,u,u^2,-q),\qquad
u_{\rm fit}[\varepsilon(E_0)]=\sqrt{\mathbf J C\mathbf J^{\mathsf T}},
$$

其中 $C$ 是 efficiency-curve fit 的完整 parameter covariance matrix。这个误差只表示给定经验模型下的曲线误差，不包含源活度的相关误差或模型选择误差。


## 作业要求

1. 用 log scale 查看完整能谱。参照标准源能谱和表中的 $E_\gamma$、$P_\gamma$，先用两个相隔较远且指认可靠的 peak 估计线性刻度，再据此寻找其余刻度线。
2. 对参与分析的孤立 peak 做 Gaussian + linear background 的 local fit，提取 centroid、$\sigma$ 和 FWHM；检查 peak-fit residual，并在完整能谱上叠加各局部 fit 及其 background。
3. 用 centroid 与参考能量建立 energy calibration，比较一次与二次函数，并根据 calibration residual 选择刻度关系。将刻度应用到完整 histogram，确认总计数不变。
4. 由同一组 peak 的 $\sigma_{ch}$ 建立 FWHM–$E_\gamma$ 曲线，拟合 $FWHM(E)=\sqrt{A+BE+CE^2}$ 并检查 residual。
5. 以局部 FWHM 为尺度，为各孤立 peak 选择 ROI 和左右 sidebands；避开邻峰后计算 $G$、$B$、$N_{\rm net}$ 及其 counting uncertainty。
6. 将两个源的活度修正到测量日期，计算 apparent full-energy peak efficiency；在 log–log 坐标上拟合 efficiency–energy relation 并检查 residual。

进度对应第 3 章。

## 实例代码

下面只展开 867.378 keV peak。第一阶段用 fit 得到 centroid 和 FWHM；第二阶段用它们定义 ROI 与 sidebands，并由 summation 求 net area。其余参考线按相同步骤处理。


<div class="code-language-switch" role="group" aria-label="Code language">
  <span>Code language:</span>
  <button type="button" data-code-language="python" aria-pressed="true">Python / PyROOT</button>
  <button type="button" data-code-language="cpp" aria-pressed="false">ROOT C++</button>
</div>

<style>
.code-language-switch { display:none; gap:.5rem; align-items:center; margin:1rem 0; }
.code-language-switch button { padding:.3rem .8rem; border:1px solid #b8b8b8; border-radius:4px; background:#fff; cursor:pointer; }
.code-language-switch button[aria-pressed="true"] { color:#fff; background:#2f6f9f; border-color:#2f6f9f; }
.pyroot-code-marker { display:none; }
.pyroot-code-marker + .highlight {
  margin:.5rem 0 1rem;
  border:1px solid #d5d5d5;
  border-radius:2px;
  background:#f7f7f7;
}
.pyroot-code-marker + .highlight pre { margin:0; padding:.75rem 1rem; overflow-x:auto; }
.pyroot-code-cell[hidden],
.jp-CodeCell .jp-Cell-inputWrapper[hidden] { display:none !important; }
</style>

<script>
document.addEventListener("DOMContentLoaded", function () {
  const buttons = document.querySelectorAll(".code-language-switch button");
  const pythonCells = Array.from(document.querySelectorAll(".pyroot-code-marker"))
    .map(function (marker) { return marker.closest(".jp-MarkdownCell"); })
    .filter(Boolean);
  pythonCells.forEach(function (cell) { cell.classList.add("pyroot-code-cell"); });
  const cppInputs = document.querySelectorAll(".jp-CodeCell .jp-Cell-inputWrapper");

  function selectLanguage(language) {
    pythonCells.forEach(function (cell) { cell.hidden = language !== "python"; });
    cppInputs.forEach(function (input) { input.hidden = language !== "cpp"; });
    buttons.forEach(function (button) {
      button.setAttribute("aria-pressed", String(button.dataset.codeLanguage === language));
    });
  }

  buttons.forEach(function (button) {
    button.addEventListener("click", function () { selectLanguage(button.dataset.codeLanguage); });
  });
  document.querySelector(".code-language-switch").style.display = "flex";
  selectLanguage("python");
});
</script>


### 读取并查看能谱

先打开文件并取得 `h0`。`%jsroot on`（PyROOT）和 `//%jsroot on`（ROOT C++）开启 notebook 中的交互式图形。完整谱使用 log scale；这一步只改变显示，不改变计数。


<div class="pyroot-code-marker"></div>

```python
import math
import ROOT

%jsroot on
ROOT.gStyle.SetOptStat(0)

# TFile.Open 打开 ROOT 文件；Get 取得其中名为 h0 的 histogram。
input_file = ROOT.TFile.Open("gamma.root", "READ")
h0 = input_file.Get("h0")

c_spectrum = ROOT.TCanvas("c_spectrum_py", "h0", 850, 480)
c_spectrum.SetLogy()
h0.SetTitle("^{152}Eu + ^{133}Ba spectrum;channel;counts / bin")
h0.GetXaxis().SetRangeUser(40, 1300)
h0.SetMinimum(0.5)
h0.Draw("hist")
c_spectrum.Draw()
c_spectrum.SaveAs("standard_source_spectrum.png")
```


In [ ]:
//%jsroot on
#include "TBox.h"
#include "TCanvas.h"
#include "TFile.h"
#include "TF1.h"
#include "TFitResultPtr.h"
#include "TGraph.h"
#include "TGraphErrors.h"
#include "TH1.h"
#include "TLegend.h"
#include "TLine.h"
#include "TMath.h"
#include "TStyle.h"
#include <algorithm>
#include <cmath>
#include <iomanip>
#include <iostream>

gStyle->SetOptStat(0);

// TFile::Open 打开 ROOT 文件；Get 取得其中名为 h0 的 histogram。
auto inputFile = TFile::Open("gamma.root", "READ");
auto h0 = dynamic_cast<TH1*>(inputFile->Get("h0"));

auto cSpectrum = new TCanvas("cSpectrum", "h0", 850, 480);
cSpectrum->SetLogy();
h0->SetTitle("^{152}Eu + ^{133}Ba spectrum;channel;counts / bin");
h0->GetXaxis()->SetRangeUser(40, 1300);
h0->SetMinimum(0.5);
h0->Draw("hist");
cSpectrum->Draw();
cSpectrum->SaveAs("standard_source_spectrum.png");


运行后生成前面的标准源测量谱：

<img src="standard_source_spectrum.png" alt="code-generated standard-source spectrum" style="max-width:58%;" />


### 第一阶段：867.378 keV peak fit

多参数 fit 从给定初值开始搜索。初值偏离数据太远时，minimizer 可能进入错误的 local minimum 或不能稳定收敛。下图说明初值影响搜索路径，并不表示需要预先知道最终答案。

<img src="../code/minimum.png" alt="local and global minima in a fit objective" style="max-width:38%;" />

先画出局部数据。sidebands 位于 visible peak 之外，并保持在 peak 附近；其 linear fit 给出 $b_0$ 与 slope 的初值。peak maximum 减去本底估计给出 $H$ 的初值，visible peak 的位置和宽度给出 $\mu$、$\sigma$ 的初值。parameter limits 用来排除负的 peak height、负的 width 等非物理解。

本例会用到 `SetParameter` 设置初值、`SetParLimits` 设置允许范围、`GetParameter` / `GetParError` 读取结果，以及 `Integral` 计算每个 bin 的模型期望。

fit option `LIRSQN` 中，`L` 选择 binned Poisson likelihood，`I` 使用函数在每个 bin 内的积分，`R` 使用 `TF1` 的局部范围，`S` 返回完整 fit result，`Q` 关闭详细输出，`N` 不自动把函数附着到 histogram 或绘图。


#### 先拟合 sidebands

本例先排除 visible peak，只用左右 sidebands 拟合直线。这个结果仅用于初始化后面的 joint fit。


<div class="pyroot-code-marker"></div>

```python
h0.GetXaxis().SetRangeUser(0.0, 2500.0)
xmin867, xmax867 = 756.7, 768.7
x0_867 = 762.7
sidebands867 = ((756.7, 759.7), (765.7, 768.7))

sideband_graph867 = ROOT.TGraphErrors()
point = 0
for low, high in sidebands867:
    for bin_number in range(h0.FindBin(low), h0.FindBin(high) + 1):
        count = h0.GetBinContent(bin_number)
        sideband_graph867.SetPoint(point, h0.GetBinCenter(bin_number), count)
        sideband_graph867.SetPointError(point, 0.0, math.sqrt(max(count, 1.0)))
        point += 1

background_seed867 = ROOT.TF1(
    "background_seed867_py", "[0]+[1]*(x-762.7)", xmin867, xmax867
)
background_seed867.SetParNames("b0", "slope")
background_seed867.SetParameters(5.8e3, 0.0)
background_result867 = sideband_graph867.Fit(background_seed867, "RSQN")

c_background867 = ROOT.TCanvas("c_background867_py", "867 sidebands", 720, 420)
sideband_graph867.SetTitle(
    "867.378 keV: linear fit to sidebands;channel;counts / bin"
)
sideband_graph867.SetMarkerStyle(20)
sideband_graph867.GetXaxis().SetLimits(xmin867, xmax867)
sideband_graph867.SetMinimum(0.0)
sideband_graph867.Draw("AP")
background_seed867.SetLineColor(ROOT.kRed + 1)
background_seed867.Draw("same")
c_background867.Draw()
c_background867.SaveAs("fit_867_background.png")
```


In [ ]:
h0->GetXaxis()->SetRangeUser(0.0, 2500.0);
double xmin867 = 756.7;
double xmax867 = 768.7;
double x0_867 = 762.7;

auto sidebandGraph867 = new TGraphErrors();
int sidebandPoint867 = 0;
const double sidebands867[2][2] = {{756.7, 759.7}, {765.7, 768.7}};
for (const auto& interval : sidebands867) {
    for (int bin = h0->FindBin(interval[0]); bin <= h0->FindBin(interval[1]); ++bin) {
        double count = h0->GetBinContent(bin);
        sidebandGraph867->SetPoint(sidebandPoint867, h0->GetBinCenter(bin), count);
        sidebandGraph867->SetPointError(
            sidebandPoint867, 0.0, std::sqrt(std::max(count, 1.0)));
        ++sidebandPoint867;
    }
}

auto backgroundSeed867 = new TF1(
    "backgroundSeed867", "[0]+[1]*(x-762.7)", xmin867, xmax867);
backgroundSeed867->SetParNames("b0", "slope");
backgroundSeed867->SetParameters(5.8e3, 0.0);
TFitResultPtr backgroundResult867 = sidebandGraph867->Fit(
    backgroundSeed867, "RSQN");

auto cBackground867 = new TCanvas("cBackground867", "867 sidebands", 720, 420);
sidebandGraph867->SetTitle(
    "867.378 keV: linear fit to sidebands;channel;counts / bin");
sidebandGraph867->SetMarkerStyle(20);
sidebandGraph867->GetXaxis()->SetLimits(xmin867, xmax867);
sidebandGraph867->SetMinimum(0.0);
sidebandGraph867->Draw("AP");
backgroundSeed867->SetLineColor(kRed + 1);
backgroundSeed867->Draw("same");
cBackground867->Draw();
cBackground867->SaveAs("fit_867_background.png");


#### Gaussian + linear background joint fit

用 sideband fit 初始化本底，再在完整区间内同时拟合 Gaussian 与 linear background。此时 `b0` 和 `slope` 没有固定，会与 peak parameters 一起由数据确定。


<div class="pyroot-code-marker"></div>

```python
model867 = "gaus(0)+[3]+[4]*(x-762.7)"
f867 = ROOT.TF1("f867_py", model867, xmin867, xmax867)
f867.SetParNames("height", "mean", "sigma", "b0", "slope")
f867.SetParameter(0, 4.5e4)                                # peak height
f867.SetParameter(1, 762.7)                               # centroid
f867.SetParameter(2, 0.8)                                 # sigma
f867.SetParameter(3, background_seed867.GetParameter(0))  # b0 from sidebands
f867.SetParameter(4, background_seed867.GetParameter(1))  # slope from sidebands
f867.SetParLimits(0, 0.0, 1.0e7)
f867.SetParLimits(1, 760.5, 765.0)
f867.SetParLimits(2, 0.2, 3.0)

result867 = h0.Fit(f867, "LIRSQN")

mean867 = f867.GetParameter(1)
sigma867 = abs(f867.GetParameter(2))
fwhm867 = 2.0 * math.sqrt(2.0 * math.log(2.0)) * sigma867

print(f"centroid = {mean867:.4f} +/- {f867.GetParError(1):.4f} channel")
print(f"sigma    = {sigma867:.4f} +/- {f867.GetParError(2):.4f} channel")
print(f"FWHM     = {fwhm867:.4f} channel")
print(f"fit status = {int(result867)}")
```


In [ ]:
auto f867 = new TF1(
    "f867", "gaus(0)+[3]+[4]*(x-762.7)", xmin867, xmax867);
f867->SetParNames("height", "mean", "sigma", "b0", "slope");
f867->SetParameter(0, 4.5e4);                              // peak height
f867->SetParameter(1, 762.7);                             // centroid
f867->SetParameter(2, 0.8);                               // sigma
f867->SetParameter(3, backgroundSeed867->GetParameter(0)); // b0 from sidebands
f867->SetParameter(4, backgroundSeed867->GetParameter(1)); // slope from sidebands
f867->SetParLimits(0, 0.0, 1.0e7);
f867->SetParLimits(1, 760.5, 765.0);
f867->SetParLimits(2, 0.2, 3.0);

TFitResultPtr result867 = h0->Fit(f867, "LIRSQN");

double mean867 = f867->GetParameter(1);
double sigma867 = std::abs(f867->GetParameter(2));
double fwhm867 = 2.0 * std::sqrt(2.0 * std::log(2.0)) * sigma867;

std::cout << std::fixed << std::setprecision(4)
          << "centroid = " << mean867 << " +/- " << f867->GetParError(1)
          << " channel\nsigma    = " << sigma867 << " +/- "
          << f867->GetParError(2) << " channel\nFWHM     = " << fwhm867
          << " channel\nfit status = " << static_cast<int>(result867) << std::endl;


#### Fit 与 residual 图

逐 bin 计算模型期望时，与 fit option `I` 一致地使用 `Integral`。上图显示 data、signal + background 与 fitted background，下图显示 residual。


<div class="pyroot-code-marker"></div>

```python
residual867 = ROOT.TGraph()
for point, bin_number in enumerate(
    range(h0.FindBin(xmin867), h0.FindBin(xmax867) + 1)
):
    low = h0.GetBinLowEdge(bin_number)
    width = h0.GetBinWidth(bin_number)
    expected = f867.Integral(low, low + width) / width
    observed = h0.GetBinContent(bin_number)
    residual867.SetPoint(
        point, h0.GetBinCenter(bin_number),
        (observed - expected) / math.sqrt(expected)
    )

c867 = ROOT.TCanvas("c867_py", "867.378 keV fit", 720, 650)
c867.Divide(1, 2)
c867.cd(1)
h867_view = h0.Clone("h867_view_py")
h867_view.GetXaxis().SetRangeUser(xmin867, xmax867)
h867_view.SetTitle("867.378 keV peak;channel;counts / bin")
h867_view.SetMinimum(0.0)
h867_view.Draw("E")
f867.SetLineColor(ROOT.kBlue + 1)
f867.Draw("same")
background867 = ROOT.TF1(
    "background867_py", "[0]+[1]*(x-762.7)", xmin867, xmax867
)
background867.SetParameters(f867.GetParameter(3), f867.GetParameter(4))
background867.SetLineColor(ROOT.kRed + 1)
background867.SetLineStyle(2)
background867.Draw("same")
legend867 = ROOT.TLegend(0.55, 0.70, 0.88, 0.88)
legend867.AddEntry(f867, "signal + background", "l")
legend867.AddEntry(background867, "fitted background", "l")
legend867.Draw()

c867.cd(2)
residual867.SetTitle("Fit residual;channel;(n-#nu)/#sqrt{#nu}")
residual867.SetMarkerStyle(20)
residual867.Draw("AP")
zero867 = ROOT.TLine(xmin867, 0.0, xmax867, 0.0)
zero867.SetLineStyle(2)
zero867.Draw()
c867.Draw()
c867.SaveAs("fit_867_linear.png")
```


In [ ]:
auto residual867 = new TGraph();
int point867 = 0;
for (int bin = h0->FindBin(xmin867); bin <= h0->FindBin(xmax867); ++bin) {
    double low = h0->GetBinLowEdge(bin);
    double width = h0->GetBinWidth(bin);
    double expected = f867->Integral(low, low + width) / width;
    double observed = h0->GetBinContent(bin);
    residual867->SetPoint(
        point867++, h0->GetBinCenter(bin),
        (observed - expected) / std::sqrt(expected));
}

auto c867 = new TCanvas("c867", "867.378 keV fit", 720, 650);
c867->Divide(1, 2);
c867->cd(1);
auto h867View = static_cast<TH1*>(h0->Clone("h867View"));
h867View->GetXaxis()->SetRangeUser(xmin867, xmax867);
h867View->SetTitle("867.378 keV peak;channel;counts / bin");
h867View->SetMinimum(0.0);
h867View->Draw("E");
f867->SetLineColor(kBlue + 1);
f867->Draw("same");
auto background867 = new TF1(
    "background867", "[0]+[1]*(x-762.7)", xmin867, xmax867);
background867->SetParameters(f867->GetParameter(3), f867->GetParameter(4));
background867->SetLineColor(kRed + 1);
background867->SetLineStyle(2);
background867->Draw("same");
auto legend867 = new TLegend(0.55, 0.70, 0.88, 0.88);
legend867->AddEntry(f867, "signal + background", "l");
legend867->AddEntry(background867, "fitted background", "l");
legend867->Draw();

c867->cd(2);
residual867->SetTitle("Fit residual;channel;(n-#nu)/#sqrt{#nu}");
residual867->SetMarkerStyle(20);
residual867->Draw("AP");
auto zero867 = new TLine(xmin867, 0.0, xmax867, 0.0);
zero867->SetLineStyle(2);
zero867->Draw();
c867->Draw();
c867->SaveAs("fit_867_linear.png");


### 第二阶段：ROI summation

用 fitted centroid 和 FWHM 定义三个区域。下面的循环只累加 bin center 落在相应窗口内的观测计数；area measurement 不再使用 Gaussian height。


<div class="pyroot-code-marker"></div>

```python
left_low867, left_high867 = mean867 - 3.0*fwhm867, mean867 - 2.0*fwhm867
roi_low867, roi_high867 = mean867 - 1.5*fwhm867, mean867 + 1.5*fwhm867
right_low867, right_high867 = mean867 + 2.0*fwhm867, mean867 + 3.0*fwhm867

left867 = right867 = gross867 = 0.0
m_left867 = m_right867 = n_roi867 = 0
for bin_number in range(h0.FindBin(left_low867), h0.FindBin(right_high867) + 1):
    x = h0.GetBinCenter(bin_number)
    count = h0.GetBinContent(bin_number)
    if left_low867 <= x <= left_high867:
        left867 += count
        m_left867 += 1
    elif roi_low867 <= x <= roi_high867:
        gross867 += count
        n_roi867 += 1
    elif right_low867 <= x <= right_high867:
        right867 += count
        m_right867 += 1

background_count867 = 0.5 * n_roi867 * (
    left867 / m_left867 + right867 / m_right867
)
net867 = gross867 - background_count867
net_variance867 = (
    gross867
    + (0.5*n_roi867/m_left867)**2 * left867
    + (0.5*n_roi867/m_right867)**2 * right867
)
net_error867 = math.sqrt(net_variance867)

print(f"G = {gross867:.0f}")
print(f"B = {background_count867:.1f}")
print(f"N_net = {net867:.1f} +/- {net_error867:.1f}")

# Draw the three counting regions and the continuum interpolated from sidebands.
left_rate867 = left867 / m_left867
right_rate867 = right867 / m_right867
left_center867 = 0.5 * (left_low867 + left_high867)
right_center867 = 0.5 * (right_low867 + right_high867)
integration_background867 = ROOT.TF1(
    "integration_background867_py", "[0]+[1]*(x-[2])",
    left_low867, right_high867
)
integration_background867.SetParameters(
    left_rate867,
    (right_rate867-left_rate867)/(right_center867-left_center867),
    left_center867
)
integration_background867.SetLineColor(ROOT.kRed + 1)
integration_background867.SetLineStyle(2)

c_integration867 = ROOT.TCanvas("c_integration867_py", "867 integration", 760, 430)
h_integration867 = h0.Clone("h_integration867_py")
h_integration867.GetXaxis().SetRangeUser(
    left_low867 - 0.4*fwhm867, right_high867 + 0.4*fwhm867
)
h_integration867.SetTitle("867.378 keV: ROI and sidebands;channel;counts / bin")
h_integration867.SetMinimum(0.0)
h_integration867.Draw("E")
ymax867 = 1.08 * h_integration867.GetMaximum()
left_box867 = ROOT.TBox(left_low867, 0.0, left_high867, ymax867)
roi_box867 = ROOT.TBox(roi_low867, 0.0, roi_high867, ymax867)
right_box867 = ROOT.TBox(right_low867, 0.0, right_high867, ymax867)
for box in (left_box867, right_box867):
    box.SetFillColorAlpha(ROOT.kAzure - 9, 0.28)
    box.Draw("same")
roi_box867.SetFillColorAlpha(ROOT.kOrange - 2, 0.25)
roi_box867.Draw("same")
h_integration867.Draw("E same")
integration_background867.Draw("same")
legend_integration867 = ROOT.TLegend(0.55, 0.68, 0.88, 0.88)
legend_integration867.AddEntry(roi_box867, "peak ROI", "f")
legend_integration867.AddEntry(left_box867, "sidebands", "f")
legend_integration867.AddEntry(integration_background867, "estimated continuum", "l")
legend_integration867.Draw()
c_integration867.Draw()
c_integration867.SaveAs("fit_867_integration.png")
```


In [ ]:
double leftLow867 = mean867 - 3.0*fwhm867;
double leftHigh867 = mean867 - 2.0*fwhm867;
double roiLow867 = mean867 - 1.5*fwhm867;
double roiHigh867 = mean867 + 1.5*fwhm867;
double rightLow867 = mean867 + 2.0*fwhm867;
double rightHigh867 = mean867 + 3.0*fwhm867;

double left867 = 0.0;
double right867 = 0.0;
double gross867 = 0.0;
int mLeft867 = 0;
int mRight867 = 0;
int nRoi867 = 0;
for (int bin = h0->FindBin(leftLow867); bin <= h0->FindBin(rightHigh867); ++bin) {
    double x = h0->GetBinCenter(bin);
    double count = h0->GetBinContent(bin);
    if (x >= leftLow867 && x <= leftHigh867) {
        left867 += count;
        ++mLeft867;
    } else if (x >= roiLow867 && x <= roiHigh867) {
        gross867 += count;
        ++nRoi867;
    } else if (x >= rightLow867 && x <= rightHigh867) {
        right867 += count;
        ++mRight867;
    }
}

double backgroundCount867 = 0.5*nRoi867
    * (left867/mLeft867 + right867/mRight867);
double net867 = gross867 - backgroundCount867;
double netVariance867 = gross867
    + std::pow(0.5*nRoi867/mLeft867, 2)*left867
    + std::pow(0.5*nRoi867/mRight867, 2)*right867;
double netError867 = std::sqrt(netVariance867);

std::cout << std::fixed << std::setprecision(0)
          << "G = " << gross867 << "\n"
          << std::setprecision(1) << "B = " << backgroundCount867 << "\n"
          << "N_net = " << net867 << " +/- " << netError867 << std::endl;

double leftRate867 = left867/mLeft867;
double rightRate867 = right867/mRight867;
double leftCenter867 = 0.5*(leftLow867 + leftHigh867);
double rightCenter867 = 0.5*(rightLow867 + rightHigh867);
auto integrationBackground867 = new TF1(
    "integrationBackground867", "[0]+[1]*(x-[2])", leftLow867, rightHigh867);
integrationBackground867->SetParameters(
    leftRate867,
    (rightRate867-leftRate867)/(rightCenter867-leftCenter867),
    leftCenter867);
integrationBackground867->SetLineColor(kRed + 1);
integrationBackground867->SetLineStyle(2);

auto cIntegration867 = new TCanvas("cIntegration867", "867 integration", 760, 430);
auto hIntegration867 = static_cast<TH1*>(h0->Clone("hIntegration867"));
hIntegration867->GetXaxis()->SetRangeUser(
    leftLow867 - 0.4*fwhm867, rightHigh867 + 0.4*fwhm867);
hIntegration867->SetTitle("867.378 keV: ROI and sidebands;channel;counts / bin");
hIntegration867->SetMinimum(0.0);
hIntegration867->Draw("E");
double ymax867 = 1.08*hIntegration867->GetMaximum();
auto leftBox867 = new TBox(leftLow867, 0.0, leftHigh867, ymax867);
auto roiBox867 = new TBox(roiLow867, 0.0, roiHigh867, ymax867);
auto rightBox867 = new TBox(rightLow867, 0.0, rightHigh867, ymax867);
leftBox867->SetFillColorAlpha(kAzure - 9, 0.28);
rightBox867->SetFillColorAlpha(kAzure - 9, 0.28);
roiBox867->SetFillColorAlpha(kOrange - 2, 0.25);
leftBox867->Draw("same");
rightBox867->Draw("same");
roiBox867->Draw("same");
hIntegration867->Draw("E same");
integrationBackground867->Draw("same");
auto legendIntegration867 = new TLegend(0.55, 0.68, 0.88, 0.88);
legendIntegration867->AddEntry(roiBox867, "peak ROI", "f");
legendIntegration867->AddEntry(leftBox867, "sidebands", "f");
legendIntegration867->AddEntry(integrationBackground867, "estimated continuum", "l");
legendIntegration867->Draw();
cIntegration867->Draw();
cIntegration867->SaveAs("fit_867_integration.png");


运行后，fit 与 integration 分别回答两个问题：

<div class="peak-example-grid">
  <figure>
    <img src="fit_867_linear.png" alt="code-generated 867 keV peak fit" />
    <figcaption>fit 提取 centroid 与 FWHM，并用 residual 检查模型。</figcaption>
  </figure>
  <figure>
    <img src="fit_867_integration.png" alt="code-generated FWHM-scaled peak integration regions" />
    <figcaption>ROI summation 减去由 sidebands 估计的 continuum，得到 $N_{\rm net}$。</figcaption>
  </figure>
</div>


## 参考结果

以下结果用于完成作业后的核对。867.378 keV 示例得到：

| quantity | result |
| --- | ---: |
| centroid | $762.6016\pm0.0015$ channel |
| $\sigma$ | $0.8180\pm0.0014$ channel |
| FWHM | $1.9263$ channel |
| gross counts $G$ | $642327$ |
| estimated background $B$ | $165432.4$ |
| net area $N_{\rm net}$ | $476894.6\pm934.8$ |

peak fit status 为 0。residual 中仍可见小的系统结构，说明 Gaussian + linear background 是初步模型；fit status 不能代替 residual 检查。

### 全部局部拟合

下面把所有用于能量刻度和 FWHM 的 local fit 叠加在完整 log-y 能谱上。蓝色实线是 signal + background，红色虚线是 fitted linear background；每组函数只画在自己的局部区间内。

<img src="../code/all_peak_fits.png" alt="all local signal-plus-linear-background fits overlaid on the gamma spectrum" style="max-width:68%;" />

使用全部刻度线后，一次能量刻度给出

$$E_\gamma\;(\mathrm{keV})\approx-39.924+1.189801\,ch,$$

线性刻度的最大绝对 calibration residual 约为 0.09 keV；二次函数改善很小，因此参考结果仍采用线性刻度。

<style>
.result-grid { display:grid; grid-template-columns:repeat(2,minmax(0,1fr)); gap:1rem; margin:1rem 0 1.5rem; }
.result-grid figure { margin:0; padding:.6rem; border:1px solid #ddd; background:#fff; }
.result-grid img { display:block; width:100%; height:auto; }
.result-grid figcaption { margin-top:.5rem; font-size:.92rem; line-height:1.4; }
@media (max-width:720px) { .result-grid { grid-template-columns:1fr; } }
</style>

### 能量刻度

<div class="result-grid">
  <figure>
    <img src="reference_calibrated_spectrum.png" alt="calibrated gamma spectrum" />
    <figcaption>将最终刻度关系应用于原始 histogram 后得到的能量谱；变换前后总计数一致。</figcaption>
  </figure>
  <figure>
    <img src="reference_energy_calibration.png" alt="energy calibration and residuals" />
    <figcaption>线性与二次刻度关系及其 calibration residual。</figcaption>
  </figure>
</div>

500 keV 处的 fitted apparent efficiency 为 $10.6488\%\pm0.0260\%$；这里的误差只来自 efficiency-curve fit 的 parameter covariance。

### 峰宽与效率

<div class="result-grid">
  <figure>
    <img src="reference_fwhm.png" alt="FWHM versus energy and residuals" />
    <figcaption>FWHM–$E_\gamma$ 曲线及 residual；经验函数为 $\sqrt{A+BE+CE^2}$。</figcaption>
  </figure>
  <figure>
    <img src="reference_efficiency.png" alt="apparent full-energy peak efficiency and residuals" />
    <figcaption>由 ROI summation 得到的 apparent full-energy peak efficiency 及相对 residual；横纵坐标均为 log scale。</figcaption>
  </figure>
</div>
